In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.data_loader import load_unsupervised_features

from src.eda.eda_utils import get_dataframe_overview

from src.clustering.clustering_preprocessing import ClusteringPreprocessingBuilder
from src.clustering.clustering_models import ClusteringModelTrainer
from src.clustering.clustering_evaluation import ClusteringEvaluator

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

The scope of this notebook is to segment customers based on credit-risk profile and financial behavior features.

Unsupervised modeling workflow uses the processed features created during preprocessing and future engineering steps.

The clustering approaches compared in this notebook will be:
* KMeans as the baseline centroid-based model
* MiniBatchKMeans as a scalable centroid-based benchmark
* Gaussian Mixture Model as a probabilistic benchmark
* BisectingKMeans as a scalable hierarchical-style centroid benchmark

#### **Unsupervised Features**

In [2]:
### loading features
# unsupervised features
unsupervised_features = load_unsupervised_features()
print(f'Unsupervised features shape: {unsupervised_features.shape}')
unsupervised_features.head()

Unsupervised features shape: (59472, 44)


,Loan To Value,Branch ID,Age,Employment Type,State,FICO Score,Number of Accounts,Number of Active Accounts,Number of Overdue Accounts,Current Balance Amount,Disbursed Amount,Instalment Amount,Number of Accounts Opened Last 6 Months,Number of Delinquencies Last 6 Months,Average Account Age,Number of Inquiries,DisbursementYear,DisbursementQuarter,DisbursementMonth,DaysSinceDisbursement,IsBureauExcluded,Legit FICO Scores,FICO Rating,FICO Rating Ordinal,HasNegativeCurrentBalance,HasZeroCurrentBalance,CurrentBalancePositiveAmount,InstalmentToDisbursedRatio,BalanceToDisbursedRatio,HasNoActiveAccounts,ActiveAccountRatio,OverdueAccountRatio,RecentlyOpenedAccountRatio,RecentlyDelinquencyAccountRatio,CreditHistoryAgeBand,AccountsPerCreditHistoryMonth,RecentlyOpenedAccountsPerCreditHistoryMonth,InquiryIntensityBand,InquiryPerCreditHistoryMonth,InquiryToAccountRatio,IsHighLTV,IsVeryHighLTV,Legit LTV,InstalmentToDisbursedAmount
0,73.23,67,33,Self employed,Nevada,598,1,1.0,1,27600,50200,1991,0,1,13,0,2018,3,9,56,0,598.0,Fair,2,0,0,1,0.039661,0.000020,0,1.0,1.0,0.0,1.0,Established,0.076923,0.0,No Inquiry,0.000000,0.000000,0,0,73.23,0.039661
1,88.48,67,25,Self employed,Nevada,305,3,0.0,0,0,0,31,0,0,8,1,2018,4,10,86,0,305.0,Poor,1,0,1,1,NaN,NaN,1,0.0,0.0,0.0,0.0,Young,0.375000,0.0,Single Inquiry,0.125000,0.333333,1,0,88.48,NaN
2,89.66,67,28,Self employed,Nevada,825,2,0.0,0,0,0,1347,0,0,21,0,2018,3,9,49,0,825.0,Exceptional,5,0,1,1,NaN,NaN,1,0.0,0.0,0.0,0.0,Established,0.095238,0.0,No Inquiry,0.000000,0.000000,1,0,89.66,NaN
3,71.89,67,29,Salaried,Nevada,17,1,1.0,0,72879,74500,0,0,0,2,0,2018,3,9,46,1,NaN,Bureau Excluded,0,0,0,1,0.000000,0.000013,0,1.0,0.0,0.0,0.0,Very Young,0.500000,0.0,No Inquiry,0.000000,0.000000,0,0,71.89,0.000000
4,89.56,67,27,Self employed,Nevada,718,1,1.0,0,-41,365384,0,0,0,56,1,2018,3,9,35,0,718.0,Good,3,1,0,0,0.000000,0.000000,0,1.0,0.0,0.0,0.0,Mature,0.017857,0.0,Single Inquiry,0.017857,1.000000,1,0,89.56,0.000000


In [3]:
### unsupervised features - dataframe overview
overview = get_dataframe_overview(df = unsupervised_features)
overview

,dtype,missing_count,missing_rate,unique_count,unique_rate
InstalmentToDisbursedRatio,float64,11120,0.1870,29617,0.4980
InstalmentToDisbursedAmount,float64,11120,0.1870,29617,0.4980
BalanceToDisbursedRatio,float64,11120,0.1870,26575,0.4468
Legit FICO Scores,float64,6522,0.1097,561,0.0094
AccountsPerCreditHistoryMonth,float64,1809,0.0304,1114,0.0187
InquiryPerCreditHistoryMonth,float64,1809,0.0304,257,0.0043
RecentlyOpenedAccountsPerCreditHistoryMonth,float64,1809,0.0304,243,0.0041
ActiveAccountRatio,float64,222,0.0037,415,0.0070
Number of Active Accounts,float64,222,0.0037,36,0.0006
Legit LTV,float64,34,0.0006,5346,0.0899


In [4]:
### unsupervised features - inspecting clustering feature groups
clustering_preprocessor = ClusteringPreprocessingBuilder(
    low_cardinality_threshold = 10,
    high_cardinality_categorical_features = ['State'],
    columns_to_exclude = ['Target']
)

# drop target variable if present in unsupervised features
X_clustering = clustering_preprocessor.clean_clustering_input(df = unsupervised_features)

# get feature groups
feature_groups = clustering_preprocessor.get_feature_groups(X = X_clustering)
feature_groups

{'numeric_features': ['Loan To Value',
  'Branch ID',
  'Age',
  'FICO Score',
  'Number of Accounts',
  'Number of Active Accounts',
  'Number of Overdue Accounts',
  'Current Balance Amount',
  'Disbursed Amount',
  'Instalment Amount',
  'Number of Accounts Opened Last 6 Months',
  'Number of Delinquencies Last 6 Months',
  'Average Account Age',
  'Number of Inquiries',
  'DisbursementYear',
  'DisbursementQuarter',
  'DisbursementMonth',
  'DaysSinceDisbursement',
  'IsBureauExcluded',
  'Legit FICO Scores',
  'FICO Rating Ordinal',
  'HasNegativeCurrentBalance',
  'HasZeroCurrentBalance',
  'CurrentBalancePositiveAmount',
  'InstalmentToDisbursedRatio',
  'BalanceToDisbursedRatio',
  'HasNoActiveAccounts',
  'ActiveAccountRatio',
  'OverdueAccountRatio',
  'RecentlyOpenedAccountRatio',
  'RecentlyDelinquencyAccountRatio',
  'AccountsPerCreditHistoryMonth',
  'RecentlyOpenedAccountsPerCreditHistoryMonth',
  'InquiryPerCreditHistoryMonth',
  'InquiryToAccountRatio',
  'IsHighLTV',


#### **Modeling**

The clustering workflow follows the same modular design as the supervised modeling workflow.

Processing pipeline steps:
* numeric imputation
* robust scaling
* one-hot encoding for low-cardinality categorical features
* frequency-encoding for high-cardinality categorical features

The clustering models are compared across different cluster numbers using internal validation metrics:
* Silhouette score: higher is better
* Calinski-Harabasz score: higher is better
* Davies-Bouldin score: lower is better
* Inertia: lower is better

In [5]:
### unsupervised model - initialize trainer
clustering_trainer = ClusteringModelTrainer(
    cluster_range = range(3, 7),
    random_state = 2,
    preprocessing_builder = clustering_preprocessor,
    evaluator = ClusteringEvaluator(
        silhouette_sample_size = 10_000,
        random_state = 2
    ),
    algorithms = [
        'kmeans',
        'minibatch_kmeans',
        'gaussian_mixture',
        'bisecting_kmeans'
    ]
)

# candidate clustering models for a sample n_clusters
sample_n_clusters = 3
candidate_clusterers = clustering_trainer.get_candidate_clusterers(n_clusters = sample_n_clusters)
candidate_clusterers

{'kmeans_k3': KMeans(n_clusters=3, random_state=2),
 'minibatch_kmeans_k3': MiniBatchKMeans(max_iter=300, n_clusters=3, random_state=2),
 'gaussian_mixture_k3': GaussianMixture(covariance_type='diag', max_iter=300, n_components=3, n_init=3,
                 random_state=2),
 'bisecting_kmeans_k3': BisectingKMeans(max_iter=150, n_clusters=3, random_state=2)}

In [6]:
### unsupervised model - train candidate clustering models using the clustering pipeline
clustering_metrics = clustering_trainer.train_all_models(
    save_models = True,
    save_metrics = True,
    save_cluster_labels = True
)
clustering_metrics

Training kmeans_k3
Training minibatch_kmeans_k3
Training gaussian_mixture_k3
Training bisecting_kmeans_k3
Training kmeans_k4
Training minibatch_kmeans_k4
Training gaussian_mixture_k4
Training bisecting_kmeans_k4
Training kmeans_k5
Training minibatch_kmeans_k5
Training gaussian_mixture_k5
Training bisecting_kmeans_k5
Training kmeans_k6
Training minibatch_kmeans_k6
Training gaussian_mixture_k6
Training bisecting_kmeans_k6


,model,n_clusters,n_observations,n_assigned_clusters,inertia,davies_bouldin_score,calinski_harabasz_score,silhouette_score,is_valid_clustering,evaluation_note,cluster_sizes,min_cluster_count,min_cluster_share,n_iter,gmm_lower_bound
0,kmeans_k3,3,59472,3,4.327176e+10,0.180749,6.955804e+06,0.998039,True,Valid clustering solution,"{0: 59450, 1: 1, 2: 21}",1,0.000017,2.0,NaN
1,bisecting_kmeans_k4,4,59472,4,2.650554e+10,0.488959,7.582902e+06,0.997951,True,Valid clustering solution,"{0: 1, 1: 59433, 2: 32, 3: 6}",1,0.000017,NaN,NaN
2,kmeans_k4,4,59472,4,2.905590e+10,0.393548,6.915579e+06,0.997914,True,Valid clustering solution,"{0: 59446, 1: 1, 2: 20, 3: 5}",1,0.000017,2.0,NaN
3,kmeans_k5,5,59472,5,2.327066e+10,0.330514,6.479715e+06,0.997845,True,Valid clustering solution,"{0: 59446, 1: 1, 2: 16, 3: 4, 4: 5}",1,0.000017,2.0,NaN
4,bisecting_kmeans_k3,3,59472,3,4.051311e+10,0.386052,7.431470e+06,0.997741,True,Valid clustering solution,"{0: 1, 1: 59433, 2: 38}",1,0.000017,NaN,NaN
5,kmeans_k6,6,59472,6,9.398129e+09,0.360890,1.285285e+07,0.997232,True,Valid clustering solution,"{0: 59407, 1: 1, 2: 16, 3: 4, 4: 5, 5: 39}",1,0.000017,5.0,NaN
6,bisecting_kmeans_k6,6,59472,6,1.618071e+10,0.687746,7.460245e+06,0.995626,True,Valid clustering solution,"{0: 1, 1: 59364, 2: 69, 3: 16, 4: 16, 5: 6}",1,0.000017,NaN,NaN
7,bisecting_kmeans_k5,5,59472,5,2.020359e+10,0.552966,7.465646e+06,0.995104,True,Valid clustering solution,"{0: 1, 1: 59364, 2: 69, 3: 32, 4: 6}",1,0.000017,NaN,NaN
8,minibatch_kmeans_k3,3,59472,3,1.014074e+13,1.934282,3.595308e+02,0.833182,True,Valid clustering solution,"{0: 168, 1: 2343, 2: 56961}",168,0.002825,1.0,NaN
9,gaussian_mixture_k4,4,59472,4,NaN,1.291140,3.345193e+06,0.565605,True,Valid clustering solution,"{0: 52391, 1: 1, 2: 16, 3: 7064}",1,0.000017,11.0,-49.068421
